# RSNA Knee: simple image baseline

This notebook is deliberately small and self-contained:

1. find the competition files;
2. read a few evenly spaced DICOM slices from one representative series;
3. train a randomly initialized CNN on the 58 complete GT studies;
4. evaluate a fixed 46/12 study split with macro ROC-AUC;
5. retrain on all GT studies and write `/kaggle/working/submission.csv`.

There are no model weights or datasets from other teams. The most useful next
experiments are changing `N_SLICES`, using several series, adding weak labels,
or replacing `SmallStudyCNN` with a stronger architecture.


In [ ]:
# Imports and the few knobs students are expected to change.
from __future__ import annotations

import json
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset


SEED = 42
IMAGE_SIZE = 64
N_SLICES = 8
EPOCHS = 35
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("device:", DEVICE)


In [ ]:
# Locate the competition mount and read the tabular metadata.
def find_data_root():
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for candidate in candidates:
        if (candidate / "train.csv").is_file() and (candidate / "train_series.csv").is_file():
            return candidate
    for candidate in Path("/kaggle/input").glob("**/train.csv"):
        if (candidate.parent / "train_series.csv").is_file():
            return candidate.parent
    raise FileNotFoundError("Could not find the competition data mount")


DATA_ROOT = find_data_root()
train_table = pd.read_csv(DATA_ROOT / "train.csv", dtype={"StudyInstanceUID": str})
test_table = pd.read_csv(DATA_ROOT / "test.csv", dtype={"StudyInstanceUID": str})
sample_submission = pd.read_csv(DATA_ROOT / "sample_submission.csv")
TARGETS = [column for column in sample_submission.columns if column != "StudyInstanceUID"]

# The competition exposes complete labels for 58 studies. All other train rows
# are kept in train.csv but have missing target values.
labeled = train_table.dropna(subset=TARGETS).copy()
if len(labeled) == 0:
    raise RuntimeError("No complete labeled studies were found")
print("data root:", DATA_ROOT)
print("complete GT studies:", len(labeled), "targets:", TARGETS)


In [ ]:
# Minimal DICOM preprocessing.
# We select one fluid-sensitive sagittal series when available, then sample
# evenly spaced slices. This is intentionally easy to replace.
def as_int(value, default=0):
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return default


def header_key(path):
    try:
        ds = pydicom.dcmread(
            str(path), stop_before_pixels=True,
            specific_tags=["InstanceNumber", "ImagePositionPatient"],
        )
        instance = getattr(ds, "InstanceNumber", None)
        position = getattr(ds, "ImagePositionPatient", None)
        if instance is not None:
            return (0, int(instance), path.name)
        if position is not None and len(position) >= 3:
            return (1, float(position[2]), path.name)
    except Exception:
        pass
    return (2, path.name)


def read_one_slice(path):
    try:
        ds = pydicom.dcmread(str(path))
        image = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        low, high = np.percentile(image, [1.0, 99.0])
        image = np.clip((image - low) / max(float(high - low), 1e-6), 0.0, 1.0)
        # A centered square keeps the code simple and avoids assuming a fixed
        # Rows/Columns resolution across studies.
        height, width = image.shape[:2]
        side = min(height, width)
        top = (height - side) // 2
        left = (width - side) // 2
        image = image[top:top + side, left:left + side]
        return cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    except Exception:
        return None


def choose_series(records):
    def score(record):
        plane = str(record.get("Anatomical_Plane", ""))
        return (
            10 * as_int(record.get("Fluid_Sensitive")),
            3 * as_int(record.get("Fat_Suppression")),
            int(plane.lower() == "sagittal"),
        )
    return max(records, key=score)


def index_series(split):
    table = pd.read_csv(DATA_ROOT / f"{split}_series.csv", dtype=str)
    grouped = {}
    for record in table.to_dict("records"):
        grouped.setdefault(str(record["StudyInstanceUID"]), []).append(record)
    return grouped


TRAIN_SERIES = index_series("train")
TEST_SERIES = index_series("test")
CACHE = Path("/kaggle/working/teaching-baseline-cache")
CACHE.mkdir(parents=True, exist_ok=True)


def load_study(split, uid, series_index):
    target = CACHE / f"{split}_{uid}.npy"
    if target.is_file():
        return np.load(target)
    records = series_index.get(str(uid), [])
    if not records:
        raise FileNotFoundError(f"No series metadata for {split}/{uid}")
    record = choose_series(records)
    folder = DATA_ROOT / f"{split}_series" / str(uid) / str(record["SeriesInstanceUID"])
    files = sorted(folder.glob("*.dcm"), key=header_key)
    if not files:
        raise FileNotFoundError(f"No DICOM files in {folder}")
    positions = np.linspace(0, len(files) - 1, N_SLICES).round().astype(int)
    slices = []
    for position in positions:
        image = read_one_slice(files[int(position)])
        if image is None:
            image = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
        slices.append(image.astype(np.float32, copy=False))
    result = np.stack(slices, axis=0)
    np.save(target, result)
    return result


def build_images(split, uids, series_index):
    started = time.time()
    output = {}
    failures = []
    # A small thread pool hides DICOM I/O latency without complicating the
    # teaching model or data loader.
    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {
            pool.submit(load_study, split, str(uid), series_index): str(uid)
            for uid in uids
        }
        for future in as_completed(futures):
            uid = futures[future]
            try:
                output[uid] = future.result()
            except Exception as error:
                failures.append({"uid": uid, "error": repr(error)})
    if failures:
        print("preprocessing failures:", failures[:3], "count=", len(failures))
        raise RuntimeError("A baseline input could not be decoded")
    print(f"{split}: {len(output)} studies, {time.time() - started:.1f}s")
    return np.stack([output[str(uid)] for uid in uids]).astype(np.float32)


gt_uids = labeled["StudyInstanceUID"].astype(str).tolist()
test_uids = test_table["StudyInstanceUID"].astype(str).tolist()
gt_images = build_images("train", gt_uids, TRAIN_SERIES)
test_images = build_images("test", test_uids, TEST_SERIES)
gt_labels = labeled[TARGETS].to_numpy(dtype=np.float32)
print("image tensors:", gt_images.shape, test_images.shape)


In [ ]:
# A small study-level CNN. It starts from random initialization.
class SmallStudyCNN(nn.Module):
    def __init__(self, n_targets):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(64, n_targets),
        )

    def forward(self, images):
        batch, slices, height, width = images.shape
        features = self.encoder(images.reshape(batch * slices, 1, height, width))
        features = features.flatten(1).reshape(batch, slices, -1).mean(dim=1)
        return self.head(features)


def macro_auc(labels, scores):
    values = []
    for column in range(labels.shape[1]):
        if len(np.unique(labels[:, column])) >= 2:
            values.append(roc_auc_score(labels[:, column], scores[:, column]))
    return float(np.mean(values)) if values else float("nan")


def make_loader(images, labels, indices, shuffle):
    x = torch.from_numpy(np.asarray(images[indices], dtype=np.float32).copy())
    y = torch.from_numpy(np.asarray(labels[indices], dtype=np.float32).copy())
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)


def predict(model, loader):
    model.eval()
    scores, labels = [], []
    with torch.no_grad():
        for images, targets in loader:
            scores.append(torch.sigmoid(model(images.to(DEVICE))).cpu().numpy())
            labels.append(targets.numpy())
    return np.concatenate(labels), np.concatenate(scores)


def fit_with_validation(train_loader, valid_loader, valid_labels):
    model = SmallStudyCNN(len(TARGETS)).to(DEVICE)
    positive = gt_labels[train_indices].sum(axis=0)
    negative = len(train_indices) - positive
    pos_weight = torch.from_numpy(negative / np.maximum(positive, 1)).float().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    best_auc = -1.0
    best_epoch = 1
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for images, targets in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(images.to(DEVICE))
            loss = criterion(logits, targets.to(DEVICE))
            loss.backward()
            optimizer.step()
        labels, scores = predict(model, valid_loader)
        current_auc = macro_auc(labels, scores)
        if current_auc > best_auc:
            best_auc = current_auc
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
            print(f"epoch={epoch:02d} loss={loss.item():.4f} valid_macro_auc={current_auc:.4f}")
    model.load_state_dict(best_state)
    print(f"best validation macro AUC={best_auc:.4f} at epoch={best_epoch}")
    return model, best_epoch, best_auc


def fit_final(images, labels, epochs):
    model = SmallStudyCNN(len(TARGETS)).to(DEVICE)
    dataset = TensorDataset(
        torch.from_numpy(np.asarray(images, dtype=np.float32).copy()),
        torch.from_numpy(np.asarray(labels, dtype=np.float32).copy()),
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    positive = labels.sum(axis=0)
    negative = len(labels) - positive
    pos_weight = torch.from_numpy(negative / np.maximum(positive, 1)).float().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    for _ in range(max(1, epochs)):
        model.train()
        for batch_images, batch_labels in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_images.to(DEVICE)), batch_labels.to(DEVICE))
            loss.backward()
            optimizer.step()
    return model


In [ ]:
# Fixed 46/12 study split for a quick, leakage-safe smoke evaluation.
rng = np.random.default_rng(SEED)
permutation = rng.permutation(len(gt_uids))
valid_size = max(12, int(round(0.2 * len(gt_uids))))
valid_indices = permutation[:valid_size]
train_indices = permutation[valid_size:]
train_loader = make_loader(gt_images, gt_labels, train_indices, shuffle=True)
valid_loader = make_loader(gt_images, gt_labels, valid_indices, shuffle=False)

validation_model, best_epoch, validation_auc = fit_with_validation(
    train_loader, valid_loader, gt_labels[valid_indices]
)
valid_gold, valid_scores = predict(validation_model, valid_loader)
metrics = {
    "train_studies": int(len(train_indices)),
    "validation_studies": int(len(valid_indices)),
    "best_epoch": int(best_epoch),
    "validation_macro_auc": float(validation_auc),
    "per_target_auc": {
        target: (float(roc_auc_score(valid_gold[:, i], valid_scores[:, i]))
                 if len(np.unique(valid_gold[:, i])) >= 2 else None)
        for i, target in enumerate(TARGETS)
    },
}
print(json.dumps(metrics, indent=2))


In [ ]:
# Retrain on all reviewed studies, predict test, and save the submission.
final_model = fit_final(gt_images, gt_labels, best_epoch)
final_model.eval()
with torch.no_grad():
    test_scores = torch.sigmoid(final_model(torch.from_numpy(test_images).to(DEVICE))).cpu().numpy()

submission = pd.DataFrame(test_scores, columns=TARGETS)
submission.insert(0, "StudyInstanceUID", test_uids)
submission.to_csv("/kaggle/working/submission.csv", index=False)
torch.save({
    "model": final_model.state_dict(),
    "targets": TARGETS,
    "image_size": IMAGE_SIZE,
    "n_slices": N_SLICES,
    "public_weights": False,
}, "/kaggle/working/baseline_model.pt")
metrics["submission_rows"] = int(len(submission))
metrics["public_weights"] = False
with open("/kaggle/working/baseline_metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
# The cache is useful while the notebook runs, but deleting it keeps the Kaggle
# output small and makes downloading the three teaching artifacts reliable.
shutil.rmtree(CACHE, ignore_errors=True)
print("saved /kaggle/working/submission.csv", submission.shape)
print(submission.head())
